# Fine-tune PaddleOCR cho tiếng Việt — Kaggle

**Trước khi chạy, bật trong Notebook settings:**

| Cài đặt | Giá trị |
|---|---|
| Accelerator | **GPU T4 x2** (script tự dùng cả 2, tự hạ về 1 nếu lỗi) |
| Internet | **On** (bắt buộc, để cài paddle + clone PaddleOCR) |
| Persistence | Files only |

**Dữ liệu:** cách tốt nhất là *Add Data → Upload* `vi_rec_100k.zip` thành một Kaggle Dataset — Kaggle tự giải nén, ảnh nằm trong `/kaggle/input/...` nên không tốn hạn mức 20GB của `/kaggle/working`. Nếu không có, script tự tải bằng `gdown`.

**Chạy quá 12h:** script tự dừng trước hạn và luôn còn checkpoint. Để train tiếp: *Save Version* → notebook mới → *Add Data* → chọn output của lần chạy trước → Run All.

---
### Notebook này khác bản cũ ở đâu

1. **Patch `adaptive_avg_pool2d(x, [1, 40])` trong backbone.** Lúc train, số time-step CTC bị ép về đúng 40 bất kể ảnh rộng bao nhiêu, trong khi lúc infer là `W/8`. Nhãn dài trung vị 57 và tối đa 80 ký tự → phần lớn mẫu có target CTC bất khả thi. Đây là nguyên nhân chính giới hạn accuracy.
2. `MAX_TEXT_LENGTH` 80 → **96**: `NRTRLabelEncode` loại bỏ âm thầm mẫu có `len ≥ max−1`.
3. **Bật lại eval trong lúc train** → có `best_accuracy` để chọn model (bản cũ đặt `eval_batch_step=[0, 1000000]` nên tắt hẳn).
4. **Tắt `RecAug.crop`** (cắt 1–8 hàng pixel trên/dưới → xén mất dấu thanh) **và `reverse`** (đảo màu, vô nghĩa với ảnh scan sách); bỏ `RecConAug`.
5. Batch 32/GPU + **AMP**, LR scale theo batch.
6. **Width của model cuối = width thắng ở ablation 2.4** (`FINAL_WIDTH = None`), để mục 2.2 và 2.4 khớp nhau trong báo cáo. Ngoại lệ duy nhất: nếu width thắng cho `T = W/8` không lớn hơn nhãn dài nhất (640 → T=80 = đúng bằng nhãn dài nhất) thì notebook tự nâng lên và in rõ lý do — vì CTC đòi `T > L`, các dòng dài nhất sẽ có loss vô cực và bị bỏ qua.
7. Tự resume qua nhiều phiên, có ngân sách thời gian cứng.


In [ ]:
# ============================================================================
# CAU HINH — chinh o day, khong sua o duoi
# ============================================================================
import time
SESSION_START = time.time()

SEED             = 2026
MAX_TEXT_LENGTH  = 96      # nhan dai nhat 80; NRTRLabelEncode loai mau co len >= max-1
IMG_HEIGHT       = 48      # PHAI la 48 (backbone yeu cau H chia het 16 -> H_feat = 3)
IGNORE_SPACE     = True    # dinh nghia acc; ghi ro trong bao cao

# --- muc 2.4 cua de: doi DUNG 1 yeu to (do rong anh) ---
ABLATION_WIDTHS  = [640, 960]
ABLATION_EPOCHS  = 1
ABLATION_SUBSET  = 30000   # dung chung cho ca 2 nhanh -> van "chi doi 1 yeu to"

# --- muc 2.2 fine-tune cuoi ---
FINAL_EPOCHS     = 8       # tang neu con ngan sach GPU
# None = dung dung width THANG o ablation 2.4 -> bao cao nhat quan:
#   2.4 ket luan width nao tot hon, 2.2 train bang chinh width do.
# De bai chi neu cap 640 vs 960, khong yeu cau va cung khong cam gia tri khac;
# nhung dat mot so thu ba (vd 1280) se lam ablation mat y nghia trong bao cao.
# Neu van muon ep mot gia tri cu the thi dat so vao day.
FINAL_WIDTH      = None
BATCH_SIZE       = 32      # moi GPU. T4 16GB @ w960 vua du. Giam ve 24 neu OOM
EVAL_PER_EPOCH   = 2       # so lan do tren tap val trong 1 epoch -> so diem tren duong cong
PRINT_BATCH_STEP = 50      # cach bao nhieu iter thi log 1 dong (loss/acc/lr/ips)
LR_AT_BS128      = 5e-4    # LR goc cua config PaddleOCR, se scale tuyen tinh
LR_BOOST         = 1.5     # fine-tune ngan -> nhinh hon mot chut
USE_AMP          = True
NUM_WORKERS      = 2

# --- ngan sach thoi gian (Kaggle cut o 12h) ---
SESSION_BUDGET_H = 11.0
USE_MULTI_GPU    = True    # tu dong fallback ve 1 GPU neu launch loi

# --- nguon ---
PADDLEOCR_REF    = "v3.3.0"
LATIN_MODEL      = "latin_PP-OCRv5_mobile_rec"
PADDLE_PKG_VER   = "3.2.1"
PADDLE_CUDA      = "cu126"          # fallback: cu118
DRIVE_ID         = "1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM"   # chi dung neu khong add dataset

def time_left():
    return SESSION_BUDGET_H * 3600 - (time.time() - SESSION_START)

def hms(s):
    s = int(max(s, 0)); return f"{s//3600}h{(s%3600)//60:02d}m{s%60:02d}s"

print(f"Ngan sach phien: {SESSION_BUDGET_H}h  |  con lai {hms(time_left())}")


## 1. Cài đặt

Đề bài lưu ý: `pip install` trên Kaggle có thể thất bại âm thầm — cell này verify bằng `pip show` rồi mới đi tiếp.

In [ ]:
# ============================================================================
# CAI DAT — Kaggle nuot loi pip am tham, nen phai verify bang `pip show`
# ============================================================================
import subprocess, sys, os

def pip(*args, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"] + (["-q"] if quiet else []) + list(args)
    return subprocess.run(cmd, capture_output=True, text=True)

def pip_version(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return None

if pip_version("paddlepaddle-gpu") is None:
    for cuda in [PADDLE_CUDA, "cu118"]:
        print(f"cai paddlepaddle-gpu=={PADDLE_PKG_VER} ({cuda}) ...")
        r = pip(f"paddlepaddle-gpu=={PADDLE_PKG_VER}",
                "-i", f"https://www.paddlepaddle.org.cn/packages/stable/{cuda}/")
        if pip_version("paddlepaddle-gpu"):
            break
        print(r.stdout[-1500:], r.stderr[-1500:])

PADDLE_PIP_VER = pip_version("paddlepaddle-gpu")
assert PADDLE_PIP_VER, (
    "pip install paddlepaddle-gpu THAT BAI.\n"
    "Kiem tra: Notebook settings -> Internet = On, Accelerator = GPU."
)
print("paddlepaddle-gpu:", PADDLE_PIP_VER)

pip("gdown", "pyyaml", "rapidfuzz", "pandas")
for p in ["gdown", "rapidfuzz", "pyyaml"]:
    assert pip_version(p) or pip_version(p.replace("pyyaml", "PyYAML")), f"thieu {p}"

# import paddle o subprocess de neu co loi CUDA thi khong giet kernel
chk = subprocess.run([sys.executable, "-c",
    "import paddle; print('PADDLE', paddle.__version__); "
    "print('GPU', paddle.device.cuda.device_count()); paddle.utils.run_check()"],
    capture_output=True, text=True)
print(chk.stdout[-2000:])
assert "PaddlePaddle is installed successfully" in chk.stdout, chk.stdout[-3000:] + chk.stderr[-3000:]

N_GPU = int([l for l in chk.stdout.splitlines() if l.startswith("GPU ")][0].split()[1])
if not USE_MULTI_GPU:
    N_GPU = min(N_GPU, 1)
print("So GPU dung:", N_GPU)


## 2. Đường dẫn và dữ liệu

In [ ]:
# ============================================================================
# DUONG DAN + TIM DU LIEU
#   Uu tien: add vi_rec_100k.zip lam Kaggle Dataset (Kaggle tu giai nen,
#   nam o /kaggle/input/... -> khong ton dung luong /kaggle/working 20GB).
#   Neu khong co -> tu tai bang gdown ve /kaggle/temp.
# ============================================================================
from pathlib import Path
import os, shutil, zipfile, subprocess, sys

ON_KAGGLE = Path("/kaggle").exists()
WORK  = Path("/kaggle/working/vi_rec") if ON_KAGGLE else Path.cwd() / "vi_rec"
SCRATCH = Path("/kaggle/temp") if ON_KAGGLE else WORK / "temp"

CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR = WORK / "configs", WORK / "output", WORK / "results"
WEIGHT_DIR, LOG_DIR = WORK / "weights", WORK / "logs"
PADDLE_DIR = SCRATCH / "PaddleOCR"
for p in [WORK, SCRATCH, CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR, WEIGHT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv"], capture_output=True, text=True).stdout)

def find_dataset_root(*roots):
    cands = []
    for root in roots:
        if not Path(root).exists():
            continue
        for f in Path(root).rglob("rec_train.txt"):
            d = f.parent
            if "__MACOSX" in d.parts:
                continue
            if all((d / n).exists() for n in
                   ["rec_val.txt", "rec_test.txt", "vi_dict.txt", "train", "val", "test"]):
                cands.append(d)
    cands.sort(key=lambda p: len(p.parts))
    return cands[0] if cands else None

DATA_DIR = find_dataset_root("/kaggle/input", SCRATCH, WORK)

if DATA_DIR is None:
    print("Khong thay dataset trong /kaggle/input -> tai bang gdown (~3.8GB)")
    import gdown
    zp = SCRATCH / "vi_rec_100k.zip"
    if not zp.exists():
        gdown.download(id=DRIVE_ID, output=str(zp), quiet=False)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(SCRATCH / "data")
    for nested in list((SCRATCH / "data").rglob("*.zip")):
        if "__MACOSX" not in nested.parts and not nested.name.startswith("._"):
            with zipfile.ZipFile(nested) as zf:
                zf.extractall(nested.parent)
    DATA_DIR = find_dataset_root(SCRATCH)

assert DATA_DIR is not None, "Khong tim thay rec_train.txt / train/ / val/ / test/"
TRAIN_FILE, VAL_FILE = DATA_DIR / "rec_train.txt", DATA_DIR / "rec_val.txt"
TEST_FILE, DICT_FILE = DATA_DIR / "rec_test.txt", DATA_DIR / "vi_dict.txt"

# checkpoint cua phien truoc (add output notebook cu lam dataset de train tiep)
RESUME_CKPT = None
for c in Path("/kaggle/input").rglob("*.pdparams") if ON_KAGGLE else []:
    if c.stem == "latest" and (c.parent / "latest.states").exists():
        RESUME_CKPT = c.with_suffix("")
        break

print("DATA_DIR   :", DATA_DIR)
print("WORK       :", WORK)
print("resume tu  :", RESUME_CKPT or "(khong co - train tu dau)")


## 3. Clone PaddleOCR và patch backbone

Đây là cell quan trọng nhất của notebook.

In [ ]:
# ============================================================================
# CLONE PaddleOCR + VA PATCH BUG QUAN TRONG NHAT
#
# rec_lcnetv3.py / rec_hgnet.py / rec_pphgnetv2.py deu ket thuc backbone bang:
#       if self.training:  x = F.adaptive_avg_pool2d(x, [1, 40])   <-- ep 40 cot
#       else:              x = F.avg_pool2d(x, [3, 2])             <-- W_in/8
#
# => LUC TRAIN so time-step CTC luon = 40 bat ke anh rong bao nhieu.
#    Nhan cua ta dai p50=57, max=80 ky tu  ->  ~73% mau co target CTC BAT KHA THI
#    (CTC doi hoi T >= L), va lech 4x so voi luc infer.
#    Config goc dung width 320 (320/8 = 40) nen bug khong lo ra.
#
# Patch: thay 40 bang W_feat//2, dung bang dung nhanh eval. Voi width 320 no
# tai tao chinh xac hanh vi cu -> backward compatible.
# ============================================================================
import subprocess, sys
from pathlib import Path

if not PADDLE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PADDLEOCR_REF,
                    "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLE_DIR)], check=True)

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PADDLE_DIR / "requirements.txt")], capture_output=True, text=True)
assert pip_version("paddlepaddle-gpu"), "requirements.txt da pha paddlepaddle-gpu, cai lai"

OLD = "x = F.adaptive_avg_pool2d(x, [1, 40])"
NEW = "x = F.adaptive_avg_pool2d(x, [1, max(1, x.shape[3] // 2)])"
patched = []
for name in ["rec_lcnetv3.py", "rec_hgnet.py", "rec_pphgnetv2.py"]:
    f = PADDLE_DIR / "ppocr/modeling/backbones" / name
    if not f.exists():
        continue
    s = f.read_text(encoding="utf-8")
    if OLD in s:
        f.write_text(s.replace(OLD, NEW), encoding="utf-8")
        patched.append(name)
    elif NEW in s:
        patched.append(name + " (da patch tu truoc)")
assert "rec_lcnetv3.py" in " ".join(patched), "KHONG patch duoc rec_lcnetv3.py"
print("Da patch:", ", ".join(patched))

PADDLEOCR_SHA = subprocess.check_output(
    ["git", "-C", str(PADDLE_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print("PaddleOCR:", PADDLEOCR_REF, PADDLEOCR_SHA)

# so time-step CTC ma cau hinh se co
for w in ABLATION_WIDTHS + ([FINAL_WIDTH] if FINAL_WIDTH else []):
    print(f"  width {w:5d} -> T = {w // 8:3d} time-step  (nhan dai nhat 80 ky tu)")


## 4. Kiểm tra dữ liệu

In [ ]:
# ============================================================================
# KIEM TRA DU LIEU + TAO CAC FILE NHAN PHU
# ============================================================================
import unicodedata, random, os
import numpy as np, pandas as pd
from IPython.display import display

def read_labels(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if line:
                rel, text = line.split("\t", 1)
                rows.append((os.path.normpath(rel), unicodedata.normalize("NFC", text)))
    return rows

train_rows, val_rows, test_rows = map(read_labels, [TRAIN_FILE, VAL_FILE, TEST_FILE])
lengths = np.array([len(t) for _, t in train_rows])
display(pd.DataFrame({"split": ["train", "val", "test"],
                      "so_mau": [len(train_rows), len(val_rows), len(test_rows)]}))
print("Do dai nhan p50/p90/p99/max:",
      np.percentile(lengths, [50, 90, 99]).round(1).tolist(), int(lengths.max()))

# NRTRLabelEncode loai mau khi len >= max_text_length - 1
assert int(lengths.max()) < MAX_TEXT_LENGTH - 1, (
    f"MAX_TEXT_LENGTH={MAX_TEXT_LENGTH} qua nho, nhan dai nhat={lengths.max()}")

dict_chars = {unicodedata.normalize("NFC", l.rstrip("\n\r"))
              for l in open(DICT_FILE, encoding="utf-8") if l.rstrip("\n\r")}
all_chars = set("".join(t for _, t in train_rows + val_rows + test_rows)) - {" "}
missing = sorted(all_chars - dict_chars)
assert not missing, f"vi_dict.txt thieu ky tu: {missing[:20]}"
print(f"vi_dict.txt phu du {len(dict_chars)} ky tu")

# kiem tra ngau nhien 300 anh cho nhanh
rng = random.Random(SEED)
for rel, _ in rng.sample(train_rows + val_rows + test_rows, 300):
    assert (DATA_DIR / rel).exists(), f"thieu anh {rel}"
print("Anh: OK (mau 300)")

# tap con cho ablation (dung CHUNG cho ca 2 nhanh) + val nho cho eval trong luc train
SUB_TRAIN = WORK / f"rec_train_sub{ABLATION_SUBSET}.txt"
VAL_SMALL = WORK / "rec_val_small.txt"
lines = [l for l in open(TRAIN_FILE, encoding="utf-8").read().splitlines() if l.strip()]
rng.shuffle(lines)
SUB_TRAIN.write_text("\n".join(lines[:ABLATION_SUBSET]) + "\n", encoding="utf-8")
vlines = [l for l in open(VAL_FILE, encoding="utf-8").read().splitlines() if l.strip()]
VAL_SMALL.write_text("\n".join(vlines[:1000]) + "\n", encoding="utf-8")
print(f"Tap ablation: {ABLATION_SUBSET} dong  |  val nho: 1000 dong")

# ty le khung anh -> chon width
try:
    from PIL import Image
    ars = []
    for rel, _ in rng.sample(test_rows, 800):
        w, h = Image.open(DATA_DIR / rel).size
        ars.append(w / h)
    ars = np.array(ars)
    print("Aspect w/h  p50/p90/p99:", np.percentile(ars, [50, 90, 99]).round(1).tolist())
    for w in ABLATION_WIDTHS + [1280, 1536]:
        print(f"  width {w:5d}: giu nguyen ty le cho {(ars <= w / IMG_HEIGHT).mean()*100:5.1f}% anh"
              f"  | T = {w//8} time-step")
except ImportError:
    pass


## 5. Hàm tiện ích

In [ ]:
# ============================================================================
# HAM TIEN ICH: chay tien trinh co deadline, doc log TRUC TIEP, infer, cham diem
# ============================================================================
import subprocess, json, os, re, sys, time, threading, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein

ENV = os.environ.copy()
ENV["FLAGS_allocator_strategy"] = "auto_growth"
ENV["PYTHONNOUSERSITE"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"        # de log hien ra ngay, khong doi buffer day

# --- doc mot dong log cua PaddleOCR --------------------------------------
# Dinh dang: [2026/08/21 10:00:00] ppocr INFO: epoch: [1/8], global_step: 100,
#            lr: 0.0005, acc: 0.12, norm_edit_dis: 0.8, loss: 1.2, ips: 64 samples/s, eta: 1:23:45
_TS_RE    = re.compile(r"^\[(\d{4}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\]")
_EPOCH_RE = re.compile(r"epoch: \[(\d+)/(\d+)\]")
_STEP_RE  = re.compile(r"global_step: (\d+)")
_KV_RE    = re.compile(r"\b([A-Za-z_][A-Za-z_0-9]*): (-?\d+\.?\d*(?:[eE][-+]?\d+)?)")
_ETA_RE   = re.compile(r"eta: (\S+)")

def _ts(line):
    m = _TS_RE.match(line)
    if not m:
        return None
    return time.mktime(time.strptime(m.group(1), "%Y/%m/%d %H:%M:%S"))

def parse_train_log(log_path):
    """Tra ve (train_df, eval_df) doc tu file log cua tools/train.py.

    train_df: moi dong log buoc train -> epoch, step, lr, loss, acc(batch), ips, eta
    eval_df : moi lan chay eval tren tap val -> epoch, step, val_acc, val_ned,
              elapsed_s (tinh tu dong log dau tien), epoch_s (thoi gian giua 2 lan eval)
    """
    log_path = Path(log_path)
    if not log_path.exists():
        return pd.DataFrame(), pd.DataFrame()
    trs, evs, t0, ep, step = [], [], None, 0, 0
    for line in log_path.read_text(errors="ignore").splitlines():
        t = _ts(line)
        if t is not None and t0 is None:
            t0 = t
        me = _EPOCH_RE.search(line)
        if me:
            ep = int(me.group(1))
        ms = _STEP_RE.search(line)
        if ms:
            step = int(ms.group(1))
        kv = dict(_KV_RE.findall(line))

        if "epoch: [" in line and "avg_batch_cost" in line:
            row = {"epoch": ep, "step": step,
                   "elapsed_s": None if (t is None or t0 is None) else round(t - t0, 1)}
            for k in ["lr", "loss", "acc", "norm_edit_dis", "ips",
                      "avg_batch_cost", "avg_reader_cost", "CTCLoss", "NRTRLoss"]:
                if k in kv:
                    row[k] = float(kv[k])
            m = _ETA_RE.search(line)
            row["eta"] = m.group(1) if m else None
            trs.append(row)

        elif "cur metric" in line:
            evs.append({"epoch": ep, "step": step,
                        "val_acc": float(kv.get("acc", "nan")),
                        "val_ned": float(kv.get("norm_edit_dis", "nan")),
                        "fps": float(kv["fps"]) if "fps" in kv else None,
                        "elapsed_s": None if (t is None or t0 is None) else round(t - t0, 1)})

    train_df, eval_df = pd.DataFrame(trs), pd.DataFrame(evs)
    if len(eval_df) and eval_df["elapsed_s"].notna().all():
        eval_df["epoch_s"] = eval_df["elapsed_s"].diff().fillna(eval_df["elapsed_s"]).round(1)
    return train_df, eval_df

# --- theo doi log truc tiep trong luc train ------------------------------
class LogTail(threading.Thread):
    """Doc log dang duoc ghi va in ra notebook: moi lan eval in ngay,
    cac buoc train in gon lai <= 1 dong / STEP_EVERY giay."""
    STEP_EVERY = 120

    def __init__(self, log_path, total_epochs, t_start):
        super().__init__(daemon=True)
        self.path, self.total, self.t_start = Path(log_path), total_epochs, t_start
        self.stop_evt, self.pos, self.last_step_print = threading.Event(), 0, 0.0
        self.n_eval, self.cur_epoch = 0, "?"

    def _emit(self, line):
        now = time.time()
        me = _EPOCH_RE.search(line)      # dong "cur metric" khong co so epoch -> nho lai
        if me:
            self.cur_epoch = me.group(1)
        if "cur metric" in line:
            kv = dict(_KV_RE.findall(line))
            self.n_eval += 1
            print(f"  [EVAL {self.n_eval}] epoch {self.cur_epoch}/{self.total}"
                  f"  val_acc {float(kv.get('acc', 0))*100:6.2f}%"
                  f"  NED {float(kv.get('norm_edit_dis', 0)):.4f}"
                  f"  | da train {hms(now - self.t_start)}"
                  f"  | ngan sach con {hms(time_left())}", flush=True)
        elif "save best model" in line or "best metric" in line:
            print("   ", line.split("INFO: ", 1)[-1].strip(), flush=True)
        elif "epoch: [" in line and "avg_batch_cost" in line:
            if now - self.last_step_print < self.STEP_EVERY:
                return
            self.last_step_print = now
            kv = dict(_KV_RE.findall(line))
            m_eta = _ETA_RE.search(line)
            print(f"    epoch {self.cur_epoch}/{self.total}"
                  f"  step {kv.get('global_step', '?')}"
                  f"  loss {float(kv.get('loss', 0)):7.4f}"
                  f"  acc(batch) {float(kv.get('acc', 0))*100:5.2f}%"
                  f"  lr {float(kv.get('lr', 0)):.2e}"
                  f"  {float(kv.get('ips', 0)):.0f} img/s"
                  f"  eta {m_eta.group(1) if m_eta else '?'}", flush=True)
        elif "Error" in line or "Traceback" in line or "out of memory" in line.lower():
            print("    !", line.strip()[-160:], flush=True)

    def _drain(self):
        if not self.path.exists():
            return
        with open(self.path, "r", encoding="utf-8", errors="ignore") as f:
            f.seek(self.pos)
            chunk = f.read()
            self.pos = f.tell()
        for line in chunk.splitlines():
            self._emit(line)

    def run(self):
        while not self.stop_evt.wait(5):
            self._drain()
        self._drain()

def run_process(args, log_name, timeout=None, watch_epochs=None):
    """Tra ve (giay, hoan_thanh). Het gio -> terminate, checkpoint van con.
    watch_epochs != None -> bat LogTail de in tien do ngay trong luc chay."""
    log_path = LOG_DIR / log_name
    start = time.time()
    tail = None
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.Popen([str(x) for x in args], cwd=PADDLE_DIR,
                             stdout=log, stderr=subprocess.STDOUT, text=True, env=ENV)
        if watch_epochs:
            tail = LogTail(log_path, watch_epochs, start)
            tail.start()
        try:
            rc = p.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            p.terminate()
            try:
                p.wait(90)
            except subprocess.TimeoutExpired:
                p.kill()
            if tail:
                tail.stop_evt.set(); tail.join(15)
            print(f"  [het ngan sach thoi gian] dung {log_name} sau {hms(time.time()-start)}")
            return time.time() - start, False
        finally:
            if tail and tail.is_alive():
                tail.stop_evt.set(); tail.join(15)
    if rc != 0:
        print("\n".join(log_path.read_text(errors="ignore").splitlines()[-60:]))
        raise RuntimeError("Loi: " + " ".join(map(str, args)))
    return time.time() - start, True

_n_gpu = [N_GPU]
def train_once(cfg_path, log_name, resume=None, timeout=None, epochs=None):
    """Thu 2 GPU truoc; neu launch chet trong 3 phut dau thi ha ve 1 GPU."""
    def build(n):
        base = ["tools/train.py", "-c", str(cfg_path)]
        if resume:
            base += ["-o", f"Global.checkpoints={resume}"]
        if n >= 2:
            return [sys.executable, "-m", "paddle.distributed.launch", "--gpus", "0,1"] + base
        return [sys.executable] + base
    try:
        return run_process(build(_n_gpu[0]), log_name, timeout, watch_epochs=epochs)
    except RuntimeError:
        if _n_gpu[0] < 2:
            raise
        print("  paddle.distributed.launch that bai -> chuyen sang 1 GPU")
        _n_gpu[0] = 1
        import yaml as _yaml
        sd = Path(_yaml.safe_load(open(cfg_path, encoding="utf-8"))["Global"]["save_model_dir"])
        if (sd / "latest.pdparams").exists():
            resume = sd / "latest"          # dung lai epoch da xong, khong train lai tu dau
        return run_process(build(1), log_name, timeout, watch_epochs=epochs)

def run_infer(cfg_path, label_file, out_txt, log_name, pretrained=None, checkpoint=None):
    args = [sys.executable, "tools/infer_rec.py", "-c", str(cfg_path), "-o",
            f"Global.infer_img={DATA_DIR}", f"Global.infer_list={label_file}",
            f"Global.save_res_path={out_txt}", "Global.distributed=False", "Global.use_amp=False"]
    if pretrained is not None:
        args.append(f"Global.pretrained_model={pretrained}")
    if checkpoint is not None:
        args.append(f"Global.checkpoints={checkpoint}")
    return run_process(args, log_name)[0]

def read_predictions(path):
    preds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            rel = os.path.normpath(os.path.relpath(parts[0], DATA_DIR))
            preds[rel] = unicodedata.normalize("NFC", parts[1])
    return preds

def metric_text(s):
    s = unicodedata.normalize("NFC", s)
    return s.replace(" ", "") if IGNORE_SPACE else s

def evaluate(label_file, pred_txt, jsonl_path=None):
    gt_rows, preds = read_labels(label_file), read_predictions(pred_txt)
    recs, dists, correct = [], [], 0
    for rel, gt in gt_rows:
        pred = preds.get(rel, "")
        a, b = metric_text(pred), metric_text(gt)
        correct += int(a == b)
        dists.append(Levenshtein.normalized_distance(a, b))
        recs.append({"image": rel.replace(os.sep, "/"), "gt": gt, "pred": pred})
    res = {"acc": correct / len(gt_rows),
           "norm_edit_dis": 1 - float(np.mean(dists)),
           "n": len(gt_rows),
           "missing_pred": sum(1 for r, _ in gt_rows if r not in preds)}
    if jsonl_path:
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return res, recs

print("helpers OK")


## 6. Trọng số pretrained và hàm sinh config

In [ ]:
# ============================================================================
# TAI TRONG SO PRETRAINED + HAM SINH CONFIG
# ============================================================================
import urllib.request, yaml, copy

BASE_CFG = PADDLE_DIR / "configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml"
LATIN_WEIGHT = WEIGHT_DIR / f"{LATIN_MODEL}_pretrained.pdparams"
if not LATIN_WEIGHT.exists():
    urllib.request.urlretrieve(
        "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/"
        f"{LATIN_MODEL}_pretrained.pdparams", LATIN_WEIGHT)
print("pretrained:", LATIN_WEIGHT.name, f"{LATIN_WEIGHT.stat().st_size/1e6:.1f} MB")

# ---------------------------------------------------------------------------
# THAY DICT: bo ppocrv5_latin_dict.txt cua model goc, dung vi_dict.txt cua de bai
#
# Dict = bang ky tu dau ra cua model. Dict latin thieu 81/233 ky tu tieng Viet
# (toan bo to hop nguyen am + dau: a. a? a^' e^` o^' u+' ...), nen model goc
# KHONG THE doan dung bat ky dong nao co dau thanh -> baseline acc ~0.
# Doi dict lam so lop dau ra thay doi => lop cuoi cua CTC/NRTR head khong khop
# shape voi pretrained. PaddleOCR bo qua cac tensor lech shape va khoi tao lai
# rieng chung; toan bo backbone + neck van duoc nap. Do la dieu ta muon.
# ---------------------------------------------------------------------------
VI_DICT = PADDLE_DIR / "ppocr/utils/dict/vi_dict.txt"
VI_DICT.write_text(DICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")

_latin_path = PADDLE_DIR / "ppocr/utils/dict/ppocrv5_latin_dict.txt"
_latin = {l.rstrip("\n") for l in open(_latin_path, encoding="utf-8") if l.rstrip("\n")} \
         if _latin_path.exists() else set()
_vi = {l.rstrip("\n") for l in open(VI_DICT, encoding="utf-8") if l.rstrip("\n")}
print(f"dict cu (latin) : {len(_latin):4d} ky tu  -> so lop dau ra {len(_latin)+2}")
print(f"dict moi (vi)   : {len(_vi):4d} ky tu  -> so lop dau ra {len(_vi)+2}")
print(f"  phu them {len(_vi - _latin)} ky tu tieng Viet ma model goc khong biet:",
      "".join(sorted(_vi - _latin))[:60])
DICT_FILE = VI_DICT          # moi config tu day tro di deu tro vao ban trong PaddleOCR

def scaled_lr():
    """LR scale tuyen tinh theo batch hieu dung; tinh lai moi lan sinh config
    vi so GPU co the tut tu 2 xuong 1 giua chung."""
    eff = BATCH_SIZE * max(_n_gpu[0], 1)
    return LR_AT_BS128 * eff / 128 * LR_BOOST, eff

LR, EFF_BS = scaled_lr()
print(f"batch/GPU={BATCH_SIZE}  x{_n_gpu[0]} GPU  -> effective {EFF_BS}, lr={LR:.2e}")

def make_config(name, width, epochs, train_label, save_dir):
    global LR, EFF_BS
    LR, EFF_BS = scaled_lr()
    cfg = yaml.safe_load(open(BASE_CFG, encoding="utf-8"))
    n_train = sum(1 for l in open(train_label, encoding="utf-8") if l.strip())
    iters_per_epoch = max(1, n_train // EFF_BS)
    # do tren tap val EVAL_PER_EPOCH lan moi epoch -> duong cong co du diem de ve
    eval_every = max(1, iters_per_epoch // max(1, EVAL_PER_EPOCH))

    g = cfg["Global"]
    g.update({
        "model_name": name, "epoch_num": int(epochs), "seed": SEED,
        "save_model_dir": str(save_dir), "save_epoch_step": 1,          # de cua muc 3
        "print_batch_step": PRINT_BATCH_STEP,
        "eval_batch_step": [0, eval_every], "cal_metric_during_train": True,
        "pretrained_model": str(LATIN_WEIGHT), "checkpoints": None,
        "character_dict_path": str(DICT_FILE), "max_text_length": MAX_TEXT_LENGTH,
        "use_space_char": True, "d2s_train_image_shape": [3, IMG_HEIGHT, width],
        "use_amp": USE_AMP, "amp_level": "O2", "amp_dtype": "float16",
        "scale_loss": 1024.0, "use_dynamic_loss_scaling": True,
        "save_res_path": str(RESULTS_DIR / f"{name}_pred.txt"),
    })

    cfg["Optimizer"]["lr"]["learning_rate"] = LR
    cfg["Optimizer"]["lr"]["warmup_epoch"] = 0 if epochs <= 1 else 1
    cfg["Metric"]["ignore_space"] = IGNORE_SPACE
    for head in cfg["Architecture"]["Head"]["head_list"]:
        if "NRTRHead" in head:
            head["NRTRHead"]["max_text_length"] = MAX_TEXT_LENGTH

    tr = cfg["Train"]["dataset"]
    tr["data_dir"] = str(DATA_DIR)
    tr["label_file_list"] = [str(train_label)]
    # bo RecConAug: nhan da dai p50=57, ghep 2 dong gan nhu luon vuot max_text_length
    tr["transforms"] = [op for op in tr["transforms"] if next(iter(op)) != "RecConAug"]
    for op in tr["transforms"]:
        if next(iter(op)) == "RecAug":
            # crop cat 1-8 hang pixel tren/duoi -> xen mat DAU THANH tieng Viet
            # reverse dao mau -> vo nghia voi anh scan sach
            op["RecAug"] = {"crop_prob": 0.0, "reverse_prob": 0.0, "tia_prob": 0.2,
                            "blur_prob": 0.3, "noise_prob": 0.3,
                            "jitter_prob": 0.3, "hsv_aug_prob": 0.3}
    tr["ext_op_transform_idx"] = 1

    sp = cfg["Train"]["sampler"]
    sp["scales"] = [[width, IMG_HEIGHT]]   # chi doi WIDTH, height phai giu 48
    sp["first_bs"] = BATCH_SIZE
    sp["fix_bs"] = True
    cfg["Train"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Train"]["loader"]["num_workers"] = NUM_WORKERS

    ev = cfg["Eval"]["dataset"]
    ev["data_dir"] = str(DATA_DIR)
    ev["label_file_list"] = [str(VAL_SMALL)]      # val nho, eval moi epoch cho re
    for op in ev["transforms"]:
        if next(iter(op)) == "RecResizeImg":
            op["RecResizeImg"]["image_shape"] = [3, IMG_HEIGHT, width]
    # eval_mode=True -> resize giu nguyen ty le, moi anh mot chieu rong => bs phai = 1
    cfg["Eval"]["loader"]["batch_size_per_card"] = 1
    cfg["Eval"]["loader"]["num_workers"] = 2

    path = CONFIG_DIR / f"{name}.yml"
    yaml.safe_dump(cfg, open(path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    return path

def make_baseline_config():
    """Model goc chay dung nhu PaddleOCR phat hanh: dict latin cua no, khong sua gi."""
    cfg = yaml.safe_load(open(BASE_CFG, encoding="utf-8"))
    cfg["Global"]["distributed"] = False
    cfg["Global"]["pretrained_model"] = str(LATIN_WEIGHT)
    cfg["Global"]["checkpoints"] = None
    cfg["Metric"]["ignore_space"] = IGNORE_SPACE
    cfg["Eval"]["dataset"]["data_dir"] = str(DATA_DIR)
    cfg["Eval"]["dataset"]["label_file_list"] = [str(TEST_FILE)]
    cfg["Eval"]["loader"]["batch_size_per_card"] = 1
    path = CONFIG_DIR / "baseline_latin.yml"
    yaml.safe_dump(cfg, open(path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    return path

def last_epoch(prefix):
    import pickle
    st = Path(str(prefix) + ".states")
    if not st.exists():
        return 0
    return int(pickle.load(open(st, "rb")).get("epoch", 0))

print("config builder OK")


## 7. Baseline — mục 2.1

In [ ]:
# ============================================================================
# 2.1 BASELINE — PaddleOCR goc, chua fine-tune, tren rec_test.txt
#
# Luu y cho bao cao: dict cua latin_PP-OCRv5_mobile_rec THIEU 81/233 ky tu
# tieng Viet (toan bo to hop nguyen am + dau: a. a? a^' e^` o^' u+'...).
# Vi vay acc ~0 la tat yeu ve mat cau truc, con norm_edit_dis van cao
# vi phan chu cai khong dau van doc dung.
# ============================================================================
BASELINE_CFG   = make_baseline_config()
BASELINE_TXT   = RESULTS_DIR / "baseline_test.txt"
BASELINE_JSONL = RESULTS_DIR / "pred_test_baseline.jsonl"

baseline_seconds = run_infer(BASELINE_CFG, TEST_FILE, BASELINE_TXT,
                             "baseline_test.log", pretrained=LATIN_WEIGHT)
baseline_metric, _ = evaluate(TEST_FILE, BASELINE_TXT, BASELINE_JSONL)
baseline_metric["inference_seconds"] = round(baseline_seconds, 1)

print("Baseline:", LATIN_MODEL, "| tap: rec_test.txt")
display(pd.DataFrame([baseline_metric]))

latin_dict = PADDLE_DIR / "ppocr/utils/dict/ppocrv5_latin_dict.txt"
if latin_dict.exists():
    lat = {l.rstrip("\n") for l in open(latin_dict, encoding="utf-8") if l.rstrip("\n")}
    thieu = [c for c in sorted(dict_chars) if c not in lat]
    print(f"\nKy tu tieng Viet KHONG co trong dict cua model goc: {len(thieu)}/{len(dict_chars)}")
    print("  ", "".join(thieu[:60]))


## 8. Ablation width 640 vs 960 — mục 2.4

In [ ]:
# ============================================================================
# 2.4 ABLATION — chi doi DUNG 1 yeu to: do rong anh dau vao 640 vs 960
#   Moi thu khac giu nguyen: cung 30k dong, 1 epoch, cung seed, cung lr,
#   cung augment, cung batch size. Do tren rec_val.txt (khong dung test).
# ============================================================================
ablation_results, ablation_curves = [], []
if time_left() < 2.0 * 3600:
    print("Khong du ngan sach cho ablation trong phien nay — bo qua, chay lai o phien sau.")
else:
    for width in ABLATION_WIDTHS:
        name = f"ablation_w{width}"
        save_dir = OUTPUT_DIR / name
        cfg = make_config(name, width, ABLATION_EPOCHS, SUB_TRAIN, save_dir)
        latest = save_dir / "latest"
        log_name = f"{name}_train.log"

        if last_epoch(latest) < ABLATION_EPOCHS:
            print(f"[train] {name}  (T = {width//8} time-step) ...")
            secs, done = train_once(cfg, log_name, timeout=max(600, time_left() - 1800),
                                    epochs=ABLATION_EPOCHS)
            (save_dir / "train_seconds.txt").write_text(str(secs))
            if not done:
                print(f"  {name} chua xong — bo qua nhanh nay")
                continue
        secs = float((save_dir / "train_seconds.txt").read_text())

        # lich su trong luc train (loss / val_acc theo step) de ve bieu do
        tr_df, ev_df = parse_train_log(LOG_DIR / log_name)
        for df, kind in [(tr_df, "train"), (ev_df, "eval")]:
            if len(df):
                d = df.copy(); d["run"], d["kind"], d["width"] = name, kind, width
                ablation_curves.append(d)

        pred_txt = RESULTS_DIR / f"{name}_val.txt"
        infer_s = run_infer(cfg, VAL_FILE, pred_txt, f"{name}_val.log", checkpoint=latest)
        m, _ = evaluate(VAL_FILE, pred_txt)
        ablation_results.append({"width": width, "time_steps": width // 8,
                                 **m, "train_seconds": round(secs, 1),
                                 "img_per_sec": round(tr_df["ips"].median(), 1) if len(tr_df) else None,
                                 "val_infer_seconds": round(infer_s, 1)})
        print(f"  -> val acc {m['acc']*100:.2f}%  NED {m['norm_edit_dis']:.4f}"
              f"  ({hms(secs)})   | con lai {hms(time_left())}")

if ablation_results:
    ablation_df = pd.DataFrame(ablation_results).sort_values("acc", ascending=False)
    ablation_df.to_csv(RESULTS_DIR / "ablation_width.csv", index=False)
    display(ablation_df)
if ablation_curves:
    ablation_curve_df = pd.concat(ablation_curves, ignore_index=True)
    ablation_curve_df.to_csv(RESULTS_DIR / "ablation_curve.csv", index=False)
    print(f"lich su ablation -> ablation_curve.csv ({len(ablation_curve_df)} dong)")


In [ ]:
# ============================================================================
# 2.4 KET LUAN — giai thich CO CHE, khong chi doc so
# ============================================================================
MAX_LABEL_LEN = int(lengths.max())

if ablation_results:
    best = max(ablation_results, key=lambda r: r["acc"])
    worst = min(ablation_results, key=lambda r: r["acc"])
    print(f"Width {best['width']} thang: val acc {best['acc']*100:.2f}% "
          f"vs {worst['acc']*100:.2f}% cua width {worst['width']}\n")
    print(f"""Vi sao:
 1. CTC can so time-step T >= do dai nhan L. Backbone cho T = W/8, nen
    width {worst['width']} -> T = {worst['time_steps']}, width {best['width']} -> T = {best['time_steps']}.
    Nhan cua bo du lieu dai p50 = {int(np.percentile(lengths,50))}, max = {MAX_LABEL_LEN} ky tu.
 2. Anh dong chu co ty le w/h trung vi ~19.7. O chieu cao {IMG_HEIGHT}, giu nguyen
    ty le can be rong ~950px. Width {worst['width']} phai NEN NGANG phan lon anh,
    lam chu bi bop lai; width {best['width']} giu duoc gan nhu nguyen ty le.
 3. Truoc khi va patch backbone, ca hai deu bi ep ve T = 40 luc train nen
    tang width chi lam anh bi nen manh hon => 640 lai thang 960. Sau khi
    patch, quan he dao lai dung nhu ly thuyet CTC du doan.""")
    WIN_WIDTH = best["width"]
else:
    WIN_WIDTH = max(ABLATION_WIDTHS)
    print("Khong co ket qua ablation trong phien nay, dung width lon nhat.")

USE_WIDTH = FINAL_WIDTH or WIN_WIDTH

# --- kiem tra CTC kha thi truoc khi train that -----------------------------
# T = W/8 phai LON HON do dai nhan; bang nhau van hong vi moi ky tu lap
# (oo, nn, ll...) doi them it nhat 1 frame blank chen giua.
T = USE_WIDTH // 8
n_bad = int((lengths > T).sum())
n_tight = int(((lengths <= T) & (lengths > T * 0.8)).sum())
print(f"\nWidth cho model cuoi: {USE_WIDTH}  (T = {T} time-step)")
print(f"  nhan dai hon T (CTC bat kha thi) : {n_bad}/{len(lengths)} dong")
print(f"  nhan dai > 0.8*T (rat chat)      : {n_tight}/{len(lengths)} dong")

if n_bad > 0 or T <= MAX_LABEL_LEN:
    alt = min([w for w in ABLATION_WIDTHS + [1280] if w // 8 > MAX_LABEL_LEN * 1.25],
              default=1280)
    print(f"\n  !! Width {USE_WIDTH} cho T = {T} <= nhan dai nhat {MAX_LABEL_LEN}."
          f"\n     Nhung dong dai nhat se co CTC loss = inf va bi bo qua khi train."
          f"\n     -> nang len {alt} (T = {alt//8}) de train duoc TOAN BO du lieu."
          f"\n     Ghi ro dieu nay trong bao cao: ablation van la {ABLATION_WIDTHS[0]} vs"
          f" {ABLATION_WIDTHS[1]} nhu de yeu cau, con model cuoi dung {alt} vi ly do tren."
          f"\n     Dat FINAL_WIDTH = {USE_WIDTH} o cell 1 neu muon giu nguyen.")
    USE_WIDTH = alt if FINAL_WIDTH is None else USE_WIDTH
    print(f"\nWidth cho model cuoi (da dieu chinh): {USE_WIDTH}  (T = {USE_WIDTH//8})")


## 9. Fine-tune cuối — mục 2.2

In [ ]:
# ============================================================================
# 2.2 FINE-TUNE CUOI — tu dong resume, tu dong dung truoc khi Kaggle cat gio
#   Trong luc train, cell nay in truc tiep:
#     - moi 2 phut: 1 dong tien do (epoch, step, loss, acc batch, lr, img/s, eta)
#     - moi lan eval: val_acc + NED + thoi gian da train + ngan sach con lai
# ============================================================================
FINAL_NAME = f"final_w{USE_WIDTH}"
FINAL_DIR  = OUTPUT_DIR / FINAL_NAME
FINAL_CFG  = make_config(FINAL_NAME, USE_WIDTH, FINAL_EPOCHS, TRAIN_FILE, FINAL_DIR)
FINAL_LATEST = FINAL_DIR / "latest"
FINAL_LOG  = "final_train.log"

resume = None
if Path(str(FINAL_LATEST) + ".pdparams").exists():
    resume = FINAL_LATEST
elif RESUME_CKPT is not None:
    resume = RESUME_CKPT
    print("resume tu dataset phien truoc:", resume)

time_file = FINAL_DIR / "train_seconds.txt"
final_train_seconds = float(time_file.read_text()) if time_file.exists() else 0.0
done_epoch = last_epoch(FINAL_LATEST) if resume else 0
print(f"Da train {done_epoch}/{FINAL_EPOCHS} epoch | ngan sach con {hms(time_left())}")

# giu lai log cua phien truoc de duong cong khong bi dut khi chay nhieu phien
HIST_DIR = FINAL_DIR / "history"; HIST_DIR.mkdir(parents=True, exist_ok=True)
if (LOG_DIR / FINAL_LOG).exists():
    import shutil as _sh
    _sh.copy(LOG_DIR / FINAL_LOG, HIST_DIR / f"train_{int(time.time())}.log")

if done_epoch < FINAL_EPOCHS:
    budget = time_left() - 45 * 60          # chua 45 phut cho phan danh gia + bao cao
    print(f"[train] {FINAL_NAME}  w={USE_WIDTH} (T={USE_WIDTH//8})  "
          f"{FINAL_EPOCHS} epoch  bs={EFF_BS}  lr={LR:.2e}  "
          f"| deadline {hms(budget)}\n")
    secs, done = train_once(FINAL_CFG, FINAL_LOG, resume=resume,
                            timeout=max(600, budget), epochs=FINAL_EPOCHS)
    final_train_seconds += secs
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    time_file.write_text(str(final_train_seconds))
    done_epoch = last_epoch(FINAL_LATEST)
    print(f"\nSau phien nay: {done_epoch}/{FINAL_EPOCHS} epoch, tong {hms(final_train_seconds)}")
    if not done:
        print("\n>>> CHUA TRAIN XONG. De chay tiep:\n"
              "    1) Save Version notebook nay (output se duoc luu lai)\n"
              "    2) Tao notebook moi, Add Data -> output cua notebook nay\n"
              "    3) Chay lai — script tu tim latest.pdparams va train tiep")

# PaddleOCR luu best_accuracy khi eval trong luc train tot len
BEST = FINAL_DIR / "best_accuracy"
FINAL_CKPT = BEST if Path(str(BEST) + ".pdparams").exists() else FINAL_LATEST
print("checkpoint dung de bao cao:", FINAL_CKPT.name)


### 9b. Số liệu từng bước train

Lấy thẳng từ log của PaddleOCR: val_acc / NED tại mỗi mốc đo, loss theo step,
thời gian mỗi epoch. Xuất ra `results/training_curve.csv`, `per_epoch.csv`,
`train_steps.csv` để bạn tự vẽ lại, kèm sẵn một bản `training_curve.png`.


In [ ]:
# ============================================================================
# SO LIEU TUNG BUOC TRAIN  ->  results/training_curve.csv + .png
#   Gop log cua tat ca cac phien (history/) lai thanh 1 duong cong lien tuc.
# ============================================================================
import matplotlib.pyplot as plt

def collect_history(log_dir, hist_dir, cur_log):
    """Noi log nhieu phien: cong don elapsed_s de truc thoi gian lien tuc."""
    logs = sorted(Path(hist_dir).glob("train_*.log"), key=lambda p: p.stat().st_mtime)
    logs.append(Path(log_dir) / cur_log)
    trs, evs, off = [], [], 0.0
    for lg in logs:
        t, e = parse_train_log(lg)
        if not len(t) and not len(e):
            continue
        for df in (t, e):
            if len(df) and "elapsed_s" in df:
                df["elapsed_s"] = df["elapsed_s"].astype(float) + off
        off = max([df["elapsed_s"].max() for df in (t, e) if len(df)] or [off])
        trs.append(t); evs.append(e)
    cat = lambda xs: pd.concat([x for x in xs if len(x)], ignore_index=True) if any(len(x) for x in xs) else pd.DataFrame()
    tr, ev = cat(trs), cat(evs)
    # bo trung lap khi mot phien bi resume va train lai vai step
    if len(tr): tr = tr.drop_duplicates("step", keep="last").sort_values("step")
    if len(ev): ev = ev.drop_duplicates("step", keep="last").sort_values("step")
    if len(ev):
        ev["phut"]   = (ev["elapsed_s"] / 60).round(1)
        ev["epoch_s"] = ev["elapsed_s"].diff().fillna(ev["elapsed_s"]).round(1)
        ev["val_acc_%"] = (ev["val_acc"] * 100).round(2)
    return tr, ev

train_hist, eval_hist = collect_history(LOG_DIR, HIST_DIR, FINAL_LOG)

if len(eval_hist) == 0:
    print("Chua co diem eval nao trong log — train them roi chay lai cell nay.")
else:
    # ---- bang so lieu de dua thang vao bao cao ----
    show = eval_hist[["epoch", "step", "val_acc_%", "val_ned", "phut", "epoch_s"]].copy()
    show.columns = ["epoch", "step", "val_acc (%)", "val_NED", "phut_da_train", "giay_tu_lan_do_truoc"]
    print(f"{len(show)} diem do tren tap val ({EVAL_PER_EPOCH} lan/epoch)")
    display(show.reset_index(drop=True))

    if len(train_hist):
        per_ep = (train_hist.groupby("epoch")
                  .agg(loss_tb=("loss", "mean"), acc_batch_tb=("acc", "mean"),
                       lr=("lr", "last"), img_per_s=("ips", "median"),
                       so_step=("step", "count"),
                       giay=("elapsed_s", lambda s: round(s.max() - s.min(), 1)))
                  .round(4))
        per_ep["val_acc_%"] = eval_hist.groupby("epoch")["val_acc_%"].max()
        per_ep["val_NED"]   = eval_hist.groupby("epoch")["val_ned"].max().round(4)
        print("\nTong hop theo epoch:")
        display(per_ep)
        per_ep.to_csv(RESULTS_DIR / "per_epoch.csv")

    eval_hist.to_csv(RESULTS_DIR / "training_curve.csv", index=False)
    if len(train_hist):
        train_hist.to_csv(RESULTS_DIR / "train_steps.csv", index=False)

    # ---- bieu do ----
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    if len(train_hist):
        ax[0].plot(train_hist["step"], train_hist["loss"], lw=.6, alpha=.35, color="C0")
        ax[0].plot(train_hist["step"], train_hist["loss"].rolling(25, min_periods=1).mean(),
                   lw=1.8, color="C0", label="loss (TB truot 25)")
        ax[0].legend()
    ax[0].set(xlabel="global_step", ylabel="train loss", title="Loss khi train")

    ax[1].plot(eval_hist["step"], eval_hist["val_acc_%"], "o-", color="C2")
    ax[1].set(xlabel="global_step", ylabel="val accuracy (%)",
              title=f"Accuracy tren val (dinh {eval_hist['val_acc_%'].max():.2f}%)")
    ax[1].grid(alpha=.3)

    ax[2].plot(eval_hist["phut"], eval_hist["val_acc_%"], "o-", color="C3", label="acc (%)")
    ax[2].plot(eval_hist["phut"], eval_hist["val_ned"] * 100, "s--", color="C4", label="NED x100")
    ax[2].set(xlabel="phut da train", ylabel="%", title="Accuracy / NED theo thoi gian")
    ax[2].legend(); ax[2].grid(alpha=.3)

    # vach mo phan cach cac epoch
    bounds = eval_hist.drop_duplicates("epoch", keep="first")
    for a, col in zip(ax, ["step", "step", "phut"]):
        for x in bounds[col]:
            a.axvline(x, color="gray", lw=.5, ls=":")
    fig.suptitle(f"{FINAL_NAME} — width {USE_WIDTH}, bs {EFF_BS}, lr {LR:.1e}", y=1.02)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "training_curve.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("da luu: training_curve.csv / train_steps.csv / per_epoch.csv / training_curve.png")


## 10. Bảng so sánh — mục 2.3

In [ ]:
# ============================================================================
# 2.3 SO SANH — mot bang duy nhat, DO TREN CUNG rec_test.txt
# ============================================================================
FINAL_VAL_TXT   = RESULTS_DIR / "final_val.txt"
FINAL_TEST_TXT  = RESULTS_DIR / "final_test.txt"
FINAL_JSONL     = RESULTS_DIR / "pred_test_finetune.jsonl"

run_infer(FINAL_CFG, VAL_FILE, FINAL_VAL_TXT, "final_val.log", checkpoint=FINAL_CKPT)
final_val_metric, _ = evaluate(VAL_FILE, FINAL_VAL_TXT)
print("rec_val.txt  :", {k: round(v, 4) if isinstance(v, float) else v
                         for k, v in final_val_metric.items()})

final_test_seconds = run_infer(FINAL_CFG, TEST_FILE, FINAL_TEST_TXT,
                               "final_test.log", checkpoint=FINAL_CKPT)
final_test_metric, final_test_records = evaluate(TEST_FILE, FINAL_TEST_TXT, FINAL_JSONL)

comparison = pd.DataFrame([
    {"Model": f"PaddleOCR goc ({LATIN_MODEL}, chua fine-tune)",
     "acc": round(baseline_metric["acc"], 4),
     "norm_edit_dis": round(baseline_metric["norm_edit_dis"], 4),
     "Thoi gian train": "0"},
    {"Model": f"Fine-tune cua nhom (width {USE_WIDTH}, {done_epoch} epoch)",
     "acc": round(final_test_metric["acc"], 4),
     "norm_edit_dis": round(final_test_metric["norm_edit_dis"], 4),
     "Thoi gian train": hms(final_train_seconds)},
])
comparison.to_csv(RESULTS_DIR / "comparison_test.csv", index=False)
print("\nTap do: rec_test.txt (n = %d)   |   acc tinh voi ignore_space=%s"
      % (final_test_metric["n"], IGNORE_SPACE))
display(comparison)


## 11. Phân tích lỗi — mục 2.5

In [ ]:
# ============================================================================
# 2.5 PHAN TICH LOI — 20 dong sai, 3 loai
# ============================================================================
import random, unicodedata
from rapidfuzz.distance import Levenshtein

TONE = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}   # huyen sac nga hoi nang

def strip_tone(s):
    """Bo 5 dau thanh, GIU dau mu/moc (e^ o+ a( ...)."""
    d = unicodedata.normalize("NFD", s)
    return unicodedata.normalize("NFC", "".join(c for c in d if c not in TONE))

def deaccent(s):
    """Ve chu cai La-tinh tran: u+ -> u, e^' -> e, d- -> d."""
    s = s.replace("\u0111", "d").replace("\u0110", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if not unicodedata.combining(c))

def classify(gt, pred):
    if gt != pred and strip_tone(gt) == strip_tone(pred):
        return "sai dau thanh"
    for op, i1, i2, j1, j2 in Levenshtein.opcodes(gt, pred):
        if op == "insert":
            # so sanh tren chu cai tran: "nguoi" -> "nguuoi" van la lap
            # du 'u' va 'u+' khac codepoint
            seg = deaccent(pred[j1:j2])
            left = deaccent(pred[j1 - 1]) if j1 > 0 else ""
            right = deaccent(pred[j2]) if j2 < len(pred) else ""
            if seg and all(c == left or c == right for c in seg):
                return "lap ky tu"
    return "sai chu cai"

wrong = [r for r in final_test_records if metric_text(r["gt"]) != metric_text(r["pred"])]
sample20 = random.Random(SEED).sample(wrong, min(20, len(wrong)))
for r in sample20:
    r["loai_loi"] = classify(r["gt"], r["pred"])

error_df = pd.DataFrame(sample20)[["image", "gt", "pred", "loai_loi"]]
error_df.to_csv(RESULTS_DIR / "error_analysis_20.csv", index=False)
pd.set_option("display.max_colwidth", 70)
display(error_df)

pct = (error_df["loai_loi"].value_counts(normalize=True) * 100).round(1)
for k in ["sai dau thanh", "sai chu cai", "lap ky tu"]:
    print(f"  {k:<16}: {pct.get(k, 0.0):5.1f} %")

# thong ke tren TOAN BO dong sai (chac chan hon 20 mau)
allc = pd.Series([classify(r["gt"], r["pred"]) for r in wrong]).value_counts(normalize=True) * 100
print(f"\nTren toan bo {len(wrong)} dong sai:")
for k, v in allc.round(1).items():
    print(f"  {k:<16}: {v:5.1f} %")

# Bo sung: ty le loi CHI khac nhau o dau phu (thanh + mu + moc), huu ich cho bao cao
only_diacritic = sum(1 for r in wrong
                     if deaccent(metric_text(r["gt"])) == deaccent(metric_text(r["pred"])))
print(f"\nTrong {len(wrong)} dong sai, {only_diacritic} dong ({only_diacritic/len(wrong)*100:.1f}%) "
      f"chi sai o DAU PHU (thanh/mu/moc), chu cai tran hoan toan dung.")

# Ghi chu cho bao cao: mot phan nhan goc cung sai
noisy = [r for r in final_test_records if any(c in r["gt"] for c in "āīūěǔ")]
print(f"\nGhi chu: {len(noisy)} dong trong test co nhan chua ky tu khong thuoc chinh ta"
      f" tieng Viet (a-macron, e-caron...), gan nhu chac chan la nhan sinh tu mot"
      f" bo OCR khac chu khong phai nguoi go. Vi du:")
for r in noisy[:2]:
    print("   gt  :", r["gt"][:80]); print("   pred:", r["pred"][:80])


## 12. Sản phẩm nộp — mục 4

In [ ]:
# ============================================================================
# MUC 4 — SAN PHAM NOP: eval.py, README.md, requirements.txt, results/
# ============================================================================
import json, shutil, platform, subprocess

REPO = WORK / "repo"; (REPO / "results").mkdir(parents=True, exist_ok=True)

(REPO / "eval.py").write_text('''"""
Cham diem OCR: acc (exact match) va norm_edit_dis.

    python eval.py results/pred_test_finetune.jsonl
    python eval.py results/pred_test_finetune.jsonl --keep-space
"""
import argparse, json, unicodedata
import numpy as np
from rapidfuzz.distance import Levenshtein

ap = argparse.ArgumentParser()
ap.add_argument("jsonl", help='moi dong: {"image":..,"gt":..,"pred":..}')
ap.add_argument("--keep-space", action="store_true",
                help="mac dinh bo space khi so sanh (ignore_space=True)")
a = ap.parse_args()

def norm(s):
    s = unicodedata.normalize("NFC", s)
    return s if a.keep_space else s.replace(" ", "")

gts, preds = [], []
with open(a.jsonl, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            gts.append(norm(r["gt"])); preds.append(norm(r["pred"]))

acc = np.mean([g == p for g, p in zip(gts, preds)])
ned = 1 - np.mean([Levenshtein.normalized_distance(g, p) for g, p in zip(gts, preds)])
cer = sum(Levenshtein.distance(g, p) for g, p in zip(gts, preds)) / sum(max(len(g), 1) for g in gts)
print(f"n              : {len(gts)}")
print(f"ignore_space   : {not a.keep_space}")
print(f"acc            : {acc:.4f}")
print(f"norm_edit_dis  : {ned:.4f}")
print(f"CER            : {cer:.4f}")
''', encoding="utf-8")

(REPO / "requirements.txt").write_text(
    f"paddlepaddle-gpu=={PADDLE_PIP_VER}\n"
    "# index: https://www.paddlepaddle.org.cn/packages/stable/cu126/\n"
    "numpy\npandas\npyyaml\nrapidfuzz\nPillow\nopencv-python\n"
    "shapely\nscikit-image\npyclipper\nlmdb\ntqdm\nalbumentations\n", encoding="utf-8")

summary = {
    "seed": SEED, "paddle": PADDLE_PIP_VER,
    "paddleocr": {"ref": PADDLEOCR_REF, "commit": PADDLEOCR_SHA},
    "patch": "rec_lcnetv3/rec_hgnet/rec_pphgnetv2: adaptive_avg_pool2d([1,40]) -> [1, W//2]",
    "pretrained": LATIN_MODEL,
    "ablation_widths": ABLATION_WIDTHS, "max_label_len": MAX_LABEL_LEN,
    "hyperparams": {"width": USE_WIDTH, "time_steps": USE_WIDTH // 8, "height": IMG_HEIGHT,
                    "max_text_length": MAX_TEXT_LENGTH, "batch_per_gpu": BATCH_SIZE,
                    "n_gpu": _n_gpu[0], "lr": LR, "epochs_done": done_epoch,
                    "epochs_target": FINAL_EPOCHS, "amp": USE_AMP,
                    "ignore_space": IGNORE_SPACE},
    "ablation": ablation_results,
    "baseline_test": baseline_metric,
    "final_val": final_val_metric,
    "final_test": final_test_metric,
    "dict": {"file": "vi_dict.txt", "n_char": len(_vi),
             "thay_the": "ppocrv5_latin_dict.txt", "n_char_cu": len(_latin),
             "ky_tu_them": len(_vi - _latin)},
    "training_curve": (eval_hist[["epoch", "step", "val_acc", "val_ned", "elapsed_s"]]
                       .to_dict("records") if len(eval_hist) else []),
    "final_train_seconds": round(final_train_seconds, 1),
    "final_test_inference_seconds": round(final_test_seconds, 1),
    "machine": {"platform": platform.platform(), "python": platform.python_version(),
                "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                      capture_output=True, text=True).stdout.strip()},
}
json.dump(summary, open(RESULTS_DIR / "summary.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

(REPO / "README.md").write_text(f"""# Fine-tune PaddleOCR cho tieng Viet

| | acc | norm_edit_dis | Thoi gian train |
|---|---|---|---|
| PaddleOCR goc ({LATIN_MODEL}) | {baseline_metric['acc']:.4f} | {baseline_metric['norm_edit_dis']:.4f} | 0 |
| Fine-tune cua nhom | {final_test_metric['acc']:.4f} | {final_test_metric['norm_edit_dis']:.4f} | {hms(final_train_seconds)} |

Do tren `rec_test.txt` (n = {final_test_metric['n']}), `ignore_space = {IGNORE_SPACE}`, seed = {SEED}.

## Moi truong
- PaddlePaddle {PADDLE_PIP_VER}, PaddleOCR {PADDLEOCR_REF} (`{PADDLEOCR_SHA}`)
- {summary['machine']['gpu'] or 'GPU'} — {summary['machine']['platform']}
- `pip install -r requirements.txt`

## Mot sua doi bat buoc trong PaddleOCR
`ppocr/modeling/backbones/rec_lcnetv3.py` (va rec_hgnet, rec_pphgnetv2) ket thuc
backbone bang `F.adaptive_avg_pool2d(x, [1, 40])` khi `self.training` — ep so
time-step CTC ve 40 bat ke anh rong bao nhieu, trong khi luc inference la `W/8`.
Nhan cua bo du lieu dai trung vi 57 va toi da 80 ky tu, nen phan lon mau co
target CTC bat kha thi (CTC doi hoi T >= L). Da thay bang `[1, W_feat // 2]`,
dung bang nhanh eval; voi width 320 mac dinh no tai tao chinh xac hanh vi cu.

## Chon do rong anh
Ablation muc 2.4 chay dung cap de bai yeu cau: **{ABLATION_WIDTHS[0]} vs {ABLATION_WIDTHS[1]}**,
chi doi do rong, moi thu khac giu nguyen. Ket qua o `results/ablation_width.csv`.
Model cuoi dung width **{USE_WIDTH}** (T = W/8 = {USE_WIDTH // 8} time-step).
Rang buoc: CTC doi T > do dai nhan, ma nhan dai nhat cua bo du lieu la {MAX_LABEL_LEN}
ky tu — nen T phai lon hon {MAX_LABEL_LEN}, tuc width > {MAX_LABEL_LEN * 8}.

## Cau hinh
width {USE_WIDTH} (T = {USE_WIDTH // 8}), height {IMG_HEIGHT}, max_text_length {MAX_TEXT_LENGTH},
batch {BATCH_SIZE}/GPU x {_n_gpu[0]}, lr {LR:.2e}, AMP {USE_AMP}, {done_epoch} epoch.
Tat `RecConAug`; `RecAug` tat `crop` va `reverse` vi crop xen mat dau thanh.

## Cach chay lai
1. Add `vi_rec_100k.zip` lam Kaggle Dataset (hoac de script tu gdown).
2. Notebook settings: Internet = On, Accelerator = GPU.
3. Run All. Neu het 12h, Save Version roi mount output do vao phien sau de train tiep.
4. `python eval.py results/pred_test_finetune.jsonl`

## Noi dung
- `results/pred_test_baseline.jsonl`, `results/pred_test_finetune.jsonl`
- `results/comparison_test.csv`, `results/ablation_width.csv`, `results/error_analysis_20.csv`
- `results/training_curve.csv` + `.png` — val_acc / NED tai tung moc train
- `results/per_epoch.csv` — loss, lr, img/s, val_acc theo tung epoch
- `results/train_steps.csv` — loss/acc/lr tung buoc (de tu ve lai bieu do)
- `logs/` — log train tho cua PaddleOCR
- `results/summary.json` — toan bo cau hinh, phien ban, thoi gian
- Checkpoint: (dien link Google Drive / HuggingFace sau khi tai ve)

## Bang phan cong
| Thanh vien | Cong viec | Dong gop |
|---|---|---|
| | | |
""", encoding="utf-8")

for f in RESULTS_DIR.glob("*"):
    if f.suffix in {".csv", ".json", ".jsonl", ".png"}:
        shutil.copy(f, REPO / "results" / f.name)
# log train tho: de ve lai bieu do sau nay ma khong can train lai
(REPO / "logs").mkdir(exist_ok=True)
for f in list(LOG_DIR.glob("*_train.log")) + list(HIST_DIR.glob("train_*.log")):
    shutil.copy(f, REPO / "logs" / f.name)

bundle = shutil.make_archive(str(WORK / "nop_bai"), "zip", root_dir=REPO)
ckpt_zip = shutil.make_archive(str(WORK / "checkpoint"), "zip", root_dir=FINAL_DIR)
print("Repo    :", REPO)
print("Bundle  :", bundle)
print("Ckpt zip:", ckpt_zip, f"({os.path.getsize(ckpt_zip)/1e6:.1f} MB)")
print("\nTai ve tu tab Output cua Kaggle, roi up checkpoint len Drive/HuggingFace.")
print("Tong thoi gian phien:", hms(time.time() - SESSION_START))
